In [1]:
from langchain.llms import ollama
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import os
import warnings

warnings.filterwarnings('ignore')

c:\Users\Rishi Kukadiya\Desktop\LLM_Medcial_Chat_Bot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
load_dotenv()

True

In [8]:
mysql_url = os.getenv("MYSQL_URL")
engine = create_engine(mysql_url)

query = "SELECT * FROM medical_data WHERE doctor = '{doctor_name}'"
df = pd.read_sql(query, engine)



In [9]:
from langchain.schema import Document
documents = [
    Document(page_content=str(row.to_dict()).lower())
        for _, row in df.iterrows()
    ]

In [10]:
import ast

# Prepare a new list of Document objects with cleaned text
processed_documents = []

for doc in documents:
    patient = ast.literal_eval(doc.page_content)  # convert string to dict
    text = (
        f"Patient Name: {patient['name']}, Age: {patient['age']}, "
        f"Gender: {patient['gender']}, Blood Type: {patient['blood type']}, "
        f"Medical Condition: {patient['medical condition']}, "
        f"Date of Admission: {patient['date of admission']}, Doctor: {patient['doctor']}, "
        f"Hospital: {patient['hospital']}, Insurance: {patient['insurance provider']}, "
        f"Billing Amount: {patient['billing amount']}, Room Number: {patient['room number']}, "
        f"Admission Type: {patient['admission type']}, Discharge Date: {patient['discharge date']}, "
        f"Medication: {patient['medication']}, Test Results: {patient['test results']}"
    )
    processed_documents.append(Document(page_content=text))


In [11]:
from langchain.embeddings import HuggingFaceBgeEmbeddings

In [12]:


embeddings=HuggingFaceBgeEmbeddings(model_name="sentence-transformers/paraphrase-MiniLM-L3-v2")


In [13]:
from langchain.vectorstores import FAISS

vector_store = FAISS.load_local(
    folder_path="patient_faiss",
    embeddings=embeddings,
    allow_dangerous_deserialization=True  # <-- allow loading your own pickle
)


In [ ]:
# FAISS.save_local(vector_store, "patient_faiss")

# Load later

In [14]:
retriever = vector_store.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 3}
)

In [15]:
retriever.invoke("who is john")

[Document(id='9471eb0b-7a6f-4a63-9f7c-ab43651f9f50', metadata={}, page_content='Patient Name: john thomas, Age: 74, Gender: female, Blood Type: a+, Medical Condition: arthritis, Date of Admission: 2021-11-13, Doctor: mark padilla, Hospital: simpson-mccall, Insurance: unitedhealthcare, Billing Amount: 29850.879728183794, Room Number: 131, Admission Type: emergency, Discharge Date: 2021-12-07, Medication: paracetamol, Test Results: inconclusive'),
 Document(id='8eb24f93-a55a-4fde-8b5e-e8143203093c', metadata={}, page_content='Patient Name: john hall, Age: 77, Gender: male, Blood Type: o-, Medical Condition: asthma, Date of Admission: 2023-10-18, Doctor: michael washington, Hospital: hunt-rodriguez, Insurance: aetna, Billing Amount: 36568.07781741183, Room Number: 486, Admission Type: elective, Discharge Date: 2023-11-07, Medication: lipitor, Test Results: abnormal'),
 Document(id='2794e47c-787e-4105-80eb-4adf4f696b3d', metadata={}, page_content='Patient Name: john travis, Age: 66, Gender

In [16]:
from langchain.llms.ollama import Ollama

llm = Ollama(model='gemma:2b', temperature=0.3)


In [17]:
from langchain.prompts import  PromptTemplate
prompt=PromptTemplate(
    template="""you are a helpful medical assistant.
    based on the following query , provide an answer using this patient database context.
    if the context is insufficient,just say don't know.
    {context}
    
    question :{question} """,
    input_variables=['context','question']
)

In [31]:
question="who is the last cancer pateint and which date is addmited"
retriver_docs=retriever.invoke(question)

In [18]:
from langchain_core.runnables import RunnableParallel,RunnableLambda,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [19]:
def format_docs(retriver_docs):
    context_text='\n\n'.join(doc.page_content for doc in retriver_docs)
    return context_text

In [20]:
parellel_chain=RunnableParallel({
    'context':retriever| RunnableLambda(format_docs),
    'question':RunnablePassthrough()
})

In [21]:
result=parellel_chain.invoke("which person age its under 20 and he has a cancer pateint")
print(result)

{'context': 'Patient Name: david brown, Age: 32, Gender: male, Blood Type: ab-, Medical Condition: cancer, Date of Admission: 2020-05-11, Doctor: john sanders, Hospital: shelton johnson, and green, Insurance: blue cross, Billing Amount: 37553.160972917416, Room Number: 256, Admission Type: emergency, Discharge Date: 2020-06-06, Medication: aspirin, Test Results: inconclusive\n\nPatient Name: jeffrey brown, Age: 80, Gender: male, Blood Type: b-, Medical Condition: cancer, Date of Admission: 2019-10-24, Doctor: jose davenport, Hospital: reed-ray, Insurance: unitedhealthcare, Billing Amount: 30363.640379615354, Room Number: 413, Admission Type: emergency, Discharge Date: 2019-11-14, Medication: paracetamol, Test Results: inconclusive\n\nPatient Name: thomas kramer, Age: 33, Gender: male, Blood Type: a+, Medical Condition: arthritis, Date of Admission: 2021-07-20, Doctor: kathleen wang, Hospital: beasley-cain, Insurance: aetna, Billing Amount: 16803.222028580883, Room Number: 454, Admissio

In [23]:
parser=StrOutputParser()


In [24]:
main_chain=parellel_chain | prompt |llm |parser

In [25]:
main_chain.invoke('What medical conditions does patient Bobby Jackson have?')

"The patient Bobby Jackson has a medical condition of cancer.\n\nThe context does not provide any other information about the patient's medical conditions."